# "THE PRICE IS RIGHT" 顶点项目

本周——基于抓取的 Amazon 数据，构建一个能根据描述预测某物价格的模型


一个能根据描述估算某物价格的模型。

# 日程安排

DAY 1：数据整理（Data Curation）  
DAY 2：数据预处理（Data Pre-processing）  
DAY 3：评估、基线、传统机器学习  
DAY 4：深度学习与 LLM  
DAY 5：微调前沿模型  

## DAY 2：数据预处理

今天我们将把产品重写为标准格式。  
LLM 很擅长这件事！


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">数据预处理 / 重写的商业价值</h2>
            <span style="color:#181;">LLM 让几年前还被认为不可能的事情变得简单。
            这种方法几乎可以应用于任何业务垂直领域，并且与我们在第 5 周使用的高级技术类似。</span>
        </td>
    </tr>
</table>

In [ ]:
# 导入：litellm 统一调多厂商模型；Batch 封装批量总结流水线

from litellm import completion
from dotenv import load_dotenv
import json
from pricer.batch import Batch
from pricer.items import Item

load_dotenv(override=True)

# 下一个单元格是你选择数据集的地方

使用 `LITE_MODE = True` 可获得免费、快速的版本，训练数据规模为 20,000

使用 `LITE_MODE =  False` 可获得强大的完整版本，训练数据规模为 800,000

## 对本实验

你可以完全跳过，直接从 Hugging Face 加载数据集：$0

你可以对 lite 数据集运行预处理：不到 $1

你可以对完整数据集运行预处理：$30

In [ ]:
# LITE_MODE=True 用小数据集快速跑通；正式跑可改 False

LITE_MODE = True

In [ ]:
# 从 Hub 拉取原始 items，合并 train/val/test 便于统一生成摘要

username = "ed-donner"
dataset = f"{username}/items_raw_lite" if LITE_MODE else f"{username}/items_raw_full"

train, val, test = Item.from_hub(dataset)

items = train + val + test

print(f"Loaded {len(items):,} items")
print(items[0])

In [ ]:
# 查看某条 item 当前的 id 字段

items[2].id

In [ ]:
# 给每个 item 一个 id

for index, item in enumerate(items):
    item.id = index

In [ ]:
# 系统提示词：要求模型输出短标题/类别/品牌/描述（英文格式请勿改）
# 后面批量生成 summary，供传统 ML 与微调使用

SYSTEM_PROMPT = """Create a concise description of a product. Respond only in this format. Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features"""

In [ ]:
# 看一条商品的 full 原文

print(items[0].full)

In [ ]:
# 用 Groq 上的开源模型试生成一条摘要，并打印 token 与费用

messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
response = completion(messages=messages, model="groq/openai/gpt-oss-20b", reasoning_effort="low")

print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")


In [ ]:
# 也可改用本地 Ollama（需本机已启动 ollama 服务）

messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
response = completion(messages=messages, model="ollama/llama3.2", api_base="http://localhost:11434")
print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")


In [ ]:
# 批量任务使用的模型名（供 jsonl body 使用）

MODEL = "openai/gpt-oss-20b"


In [ ]:
# 把单条 item 编成 Batch API 需要的 jsonl 行（custom_id + messages）

def make_jsonl(item):
    body = {"model": MODEL, "messages": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": item.full}], "reasoning_effort": "low"}
    line = {"custom_id": str(item.id), "method": "POST", "url": "/v1/chat/completions", "body": body}
    return json.dumps(line)

In [ ]:
# 确认 item 内容

items[0]

In [ ]:
# 预览一行 jsonl 长什么样

make_jsonl(items[0])

In [ ]:
# 把一段区间的 item 写成 jsonl 文件，准备上传批量任务

def make_file(start, end, filename):
    batch_file = filename
    with open(batch_file, "w", encoding="utf-8") as f:
        for i in range(start, end):
            f.write(make_jsonl(items[i]))
            f.write("\n")

In [ ]:
# 先写 0–1000 条做小批量试验

make_file(0, 1000, "jsonl/0_1000.jsonl")

In [ ]:
# 创建 Groq 客户端（批量 API）

import os
from groq import Groq

groq = Groq(api_key=os.environ.get("GROQ_API_KEY"))

In [ ]:
# 上传 jsonl 输入文件

with open("jsonl/0_1000.jsonl", "rb", encoding="utf-8") as f:
    response = groq.files.create(file=f, purpose="batch")
response

In [ ]:
# 记下上传后的 file_id

file_id = response.id
file_id

In [ ]:
# 创建 24 小时窗口的批量补全任务

response = groq.batches.create(completion_window="24h", endpoint="/v1/chat/completions", input_file_id=file_id)
response

In [ ]:
# 查询批量任务状态（需等到 completed）

result = groq.batches.retrieve(response.id)
result

In [ ]:
# 下载批量结果到本地 jsonl

response = groq.files.content(result.output_file_id)
response.write_to_file("jsonl/batch_results.jsonl")

In [ ]:
# 解析结果：按 custom_id 写回 items[id].summary

with open("jsonl/batch_results.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        json_line = json.loads(line)
        id = int(json_line["custom_id"])
        summary = json_line["response"]["body"]["choices"][0]["message"]["content"]
        items[id].summary = summary


In [ ]:
# 对比：原 full 文本

print(items[0].full)

In [ ]:
# 对比：生成的 summary（若该索引已有结果）

print(items[1000].summary)

## 我已把完全相同的逻辑放进一个 Batch 类

- 将 items 分成每组 1,000 个
- 为每一组启动批次
- 允许我们在完成时监控并收集结果

## 成本

对我来说，使用 Groq——Lite 数据集不到 $1，大数据集不到 $30

但你不必花任何钱！在下一个实验中，你可以加载我预处理好的结果

In [ ]:
# 使用课程封装：为全部 items 创建批量任务文件

Batch.create(items, LITE_MODE)

In [ ]:
# 提交并运行批量任务（耗时/费用取决于数据量）

Batch.run()

In [ ]:
# 拉取结果并写回 summary 字段

Batch.fetch()

In [ ]:
# 检查是否还有 summary 为空的样本

for index, item in enumerate(items):
    if not item.summary:
        print(index)

In [ ]:
# 抽查一条摘要

print(items[10234].summary)

In [ ]:
# 移除我们在 Hub 上不需要的字段

for item in items:
    item.full = None
    item.id = None

## 把最终数据集推送到 Hub

若为 lite 模式，我们只推送 lite 数据集

若为 full 模式，我们会推送两个数据集（以防你之后决定使用 lite）

In [ ]:
# 划分数据集并推送到 Hub（lite / full）
# 推送前一格已清空 full 与 id，只保留 summary 等必要字段

username = "ed-donner"
full = f"{username}/items_full"
lite = f"{username}/items_lite"

if LITE_MODE:
    train = items[:20_000]
    val = items[20_000:21_000]
    test = items[21_000:]
    Item.push_to_hub(lite, train, val, test)
else:
    train = items[:800_000]
    val = items[800_000:810_000]
    test = items[810_000:]
    Item.push_to_hub(full, train, val, test)

    train_lite = train[:20_000]
    val_lite = val[:1_000]
    test_lite = test[:1_000]
    Item.push_to_hub(lite, train_lite, val_lite, test_lite)

## 它们在这里！

https://huggingface.co/datasets/ed-donner/items_lite

https://huggingface.co/datasets/ed-donner/items_full
